In [1]:
import json
from pathlib import Path
import pandas as pd

def summarize_all_refs(directory_path):
    base_path = Path(directory_path)
    
    if not base_path.exists():
        print(f"❌ 错误：路径不存在 -> {directory_path}")
        return

    # 1. 扫描目录下所有 json 文件
    json_files = list(base_path.glob("*.json"))
    print(f"📂 发现 {len(json_files)} 个 REF 聚类文件，正在汇总分析...\n")

    global_entity_stats = {}

    for file in json_files:
        try:
            with open(file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                for cid, words in data.items():
                    # 过滤逻辑：去掉短噪声和纯数字
                    valid_words = [str(w).strip() for w in words if len(str(w)) >= 3 and not str(w).isdigit()]
                    if not valid_words: continue
                    
                    # 取最短词作为 ID (通常是原形)
                    rep = min(valid_words, key=len)
                    
                    if rep not in global_entity_stats:
                        global_entity_stats[rep] = {"count": 0, "max_cluster": 0, "variants": set()}
                    
                    global_entity_stats[rep]["count"] += 1 # 该实体在多少个文件中出现
                    global_entity_stats[rep]["max_cluster"] = max(global_entity_stats[rep]["max_cluster"], len(words))
                    global_entity_stats[rep]["variants"].update(valid_words[:5])
        except Exception as e:
            print(f"⚠️ 跳过文件 {file.name} (读取错误)")

    # 2. 转换为 DataFrame 进行总结
    summary_list = []
    for name, stats in global_entity_stats.items():
        summary_list.append({
            "Entité (搜索词)": name,
            "Occurrence (出现文件数)": stats["count"],
            "Max Cluster Size (最大簇容量)": stats["max_cluster"],
            "Exemples": ", ".join(list(stats["variants"])[:3])
        })

    df = pd.DataFrame(summary_list)
    # 优先排序：出现文件数多（说明算法普适性强）且 簇容量大（说明变体丰富）
    df = df.sort_values(by=["Occurrence (出现文件数)", "Max Cluster Size (最大簇容量)"], ascending=False)

    print("📋 === 全量实验推荐测试清单 (TOP 20) ===")
    #print(df.head(20).to_string(index=False))
    print(df.to_string(index=False)) # 去掉 .head(20)，使用 to_string 打印全部
    
   

if __name__ == "__main__":
    # 使用你的文件夹路径
    ref_folder = "/Users/zhengruixing/Desktop/mini-corpus/corpus_en/AINSWORTH/AINSWORTH_REF/"
    summarize_all_refs(ref_folder)

📂 发现 9 个 REF 聚类文件，正在汇总分析...

📋 === 全量实验推荐测试清单 (TOP 20) ===
                                               Entité (搜索词)  Occurrence (出现文件数)  Max Cluster Size (最大簇容量)                                                                                                                                         Exemples
                                                        And                   5                       528                                                             so\n     he goes to an agent, Hyde Park by Stanhope Gate, not an inch of his stature
                                                  Rougemont                   5                        16                                                                                           Rougemont authoritatively, Rougemont found, Rougemont’
                                                       that                   5                        10                                                                                   

In [9]:
import json
from pathlib import Path
import pandas as pd

def summarize_each_work_separately(root_path):
    base_path = Path(root_path)
    
    if not base_path.exists():
        print(f"❌ 错误：路径不存在 -> {root_path}")
        return

    # 1. 初始化统计计数器
    count_en = 0
    count_fr = 0

    # 递归查找所有包含 JSON 的 REF 文件夹
    ref_folders = list(base_path.rglob("*_REF"))
    
    if not ref_folders:
        print("📂 未发现任何包含 '_REF' 的作品文件夹。")
        return

    # 按路径排序，保证打印输出更有条理
    ref_folders.sort()

    for ref_dir in ref_folders:
        # 识别作品所属的语料库语言
        is_en = "corpus_en" in str(ref_dir)
        is_fr = "corpus_fr" in str(ref_dir)
        
        # 记录统计数量
        if is_en: count_en += 1
        if is_fr: count_fr += 1

        work_name = ref_dir.parent.name
        json_files = list(ref_dir.glob("*.json"))
        
        if not json_files: continue
        
        work_entity_stats = {}
        
        # 2. 分析该作品的所有文件
        for file in json_files:
            try:
                with open(file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for cid, words in data.items():
                        # 过滤逻辑：首字母大写聚焦实体
                        valid_words = [str(w).strip() for w in words if len(str(w)) >= 3 
                                       and not str(w).isdigit()
                                       and str(w)[0].isupper()]
                        if not valid_words: continue
                        
                        rep = min(valid_words, key=len)
                        if rep not in work_entity_stats:
                            work_entity_stats[rep] = {"count": 0, "max_cluster": 0, "variants": set()}
                        
                        work_entity_stats[rep]["count"] += 1 
                        work_entity_stats[rep]["max_cluster"] = max(work_entity_stats[rep]["max_cluster"], len(words))
                        work_entity_stats[rep]["variants"].update(valid_words[:5])
            except:
                continue

        # 3. 打印该作品的结果
        summary_list = []
        for name, stats in work_entity_stats.items():
            summary_list.append({
                "Entité (搜索词)": name,
                "Occurrence": stats["count"],
                "Max Cluster Size": stats["max_cluster"],
                "Exemples": ", ".join(list(stats["variants"])[:3])
            })

        df = pd.DataFrame(summary_list)
        if df.empty: continue
        
        df = df.sort_values(by=["Occurrence", "Max Cluster Size"], ascending=False)
        
        lang_tag = "[EN]" if is_en else "[FR]"
        print(f"\n📖 === 作品 {lang_tag}: {work_name} (TOP 20) ===")
        print(df.head(20).to_string(index=False))
        print("-" * 80)

    # 4. 打印最终统计汇总
    print(f"\n✅ 汇总报告：")
    print(f"   - corpus_en (英语作品) 共分析了: {count_en} 部")
    print(f"   - corpus_fr (法语作品) 共分析了: {count_fr} 部")
    print(f"   - 总计分析作品: {count_en + count_fr} 部")

if __name__ == "__main__":
    # 指向你的 mini-corpus 总目录
    root_folder = "/Users/zhengruixing/Desktop/mini-corpus/"
    summarize_each_work_separately(root_folder)


📖 === 作品 [EN]: AINSWORTH (TOP 20) ===
Entité (搜索词)  Occurrence  Max Cluster Size                                                                     Exemples
         I’m           6               494                                              Even if I’m catched, I’m, I’m a
         And           5               528                                                    Millbank, Strange, Strand
         Her           5                95                         Her, Auriol understand that he should give her, Here
   Rougemont           5                16                       Rougemont authoritatively, Rougemont found, Rougemont’
        That           5                10                                                       At that, Athanor, That
       Their           5                 8                                                                 Their, These
        Your           5                 6                               The coarsest ribaldry assailed your ears, Your
 